# Reference Comparison And Functional Enrichment For Mature ChP Subtypes

This notebook continues from the mature choroid plexus analysis. The previous notebook isolated mature ChP-like epithelial cells, reclustered them using PCs 1-12, and scored stress, mitochondrial, dark ChP, light/ciliated ChP, myoepithelial-like, and barrier/transport programs.

The goal here is to move from **cell-state discovery** to **biological interpretation**. We will summarize expression by subcluster/sample, extract marker genes, perform functional enrichment, compare dark and light ChP-like states, and evaluate whether mature ChP subclusters shift across organoid stages.

Because the input is a processed SCT-scaled matrix rather than raw counts, pseudobulk values here should be interpreted as **aggregate expression summaries for exploration**, not as raw-count differential expression suitable for DESeq2/edgeR.

## Analysis Plan

1. **Load mature ChP object**: start from the saved mature ChP AnnData object from notebook 03.
2. **Pseudobulk generation**: aggregate expression by sample and/or Leiden subcluster to reduce single-cell noise and compare cell states at the group level.
3. **Marker extraction**: recover marker genes for mature ChP subclusters and organize candidate genes for interpretation.
4. **Functional enrichment**: test whether marker genes are enriched for biological processes such as mitochondrial activity, transport, ciliation, epithelial barrier function, RNA processing, or retinoic-acid-related biology.
5. **Dark vs light subtype comparison**: compare dark/mitochondria-rich and light/ciliated ChP-like states using marker scores and marker genes.
6. **Sample-stage composition analysis**: ask whether subclusters are enriched at D27, D46, or D53.
7. **Interpretation**: summarize what the analyses suggest, what remains uncertain, and what should be tested against external human/mouse references later.

## 1. Load Mature ChP Object

Biological reasoning: the mature ChP subset is the relevant cell population for asking whether organoid ChP epithelium contains specialized subtypes. Loading the saved object ensures that all previous annotations, UMAP coordinates, Leiden subclusters, and signature scores are carried forward consistently.

Expected input:

```text
data/processed/mature_chp_subclustered_stress_mito.h5ad
```

If this file does not exist, run notebook `03_mature_chp_subclustering_stress_mito.ipynb` through the save step first.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, frameon=False)

PROJECT_DIR = Path("/Users/Princess/Documents/Manju's Research Portfolio")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURE_DIR = PROJECT_DIR / "results" / "figures"
TABLE_DIR = PROJECT_DIR / "results" / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

MATURE_ADATA_PATH = PROCESSED_DIR / "mature_chp_subclustered_stress_mito.h5ad"
print("Mature ChP object:", MATURE_ADATA_PATH)
print("Exists:", MATURE_ADATA_PATH.exists())

In [ ]:
if not MATURE_ADATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {MATURE_ADATA_PATH}. Run notebook 03 through the save step first."
    )

adata_mature = sc.read_h5ad(MATURE_ADATA_PATH)
print(adata_mature)
print("obs columns:")
print(list(adata_mature.obs.columns))

display(adata_mature.obs.head())

### Cell-Specific Checkpoint

Before analysis, confirm that this object contains only mature ChP-like cells from ChP organoid samples. It should not include telencephalon-reference cells if notebook 03 used the stricter mature ChP selection rule.

This matters biologically because the goal is no longer to distinguish ChP from telencephalon. The goal is to resolve heterogeneity **within mature ChP epithelium**.

In [ ]:
required_columns = ["sample", "leiden"]
missing_columns = [col for col in required_columns if col not in adata_mature.obs.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

print("Sample composition:")
display(adata_mature.obs["sample"].astype(str).value_counts())

print("Mature ChP subcluster composition:")
display(adata_mature.obs["leiden"].astype(str).value_counts().sort_index())

## 2. Pseudobulk Generation

Biological reasoning: single-cell data are noisy because each cell captures only part of its transcriptome. Pseudobulk summarizes expression across biologically meaningful groups, such as a Leiden subcluster within a sample. This makes it easier to compare mature ChP states and stage-associated patterns.

Here, pseudobulk means averaging processed expression values across cells in each group. Since the matrix is SCT-scaled, these are **not raw UMI pseudobulk counts**. They are useful for exploratory visualization and correlation, but not for raw-count statistical differential expression.

In [ ]:
def make_mean_pseudobulk(adata, groupby, layer=None):
    """Return mean expression per group as a DataFrame: groups x genes."""
    group_labels = adata.obs[groupby].astype(str)
    X = adata.layers[layer] if layer is not None else adata.X

    rows = []
    index = []
    for group in sorted(group_labels.unique()):
        mask = (group_labels == group).to_numpy()
        mean_values = np.asarray(X[mask].mean(axis=0)).ravel()
        rows.append(mean_values)
        index.append(group)

    return pd.DataFrame(rows, index=index, columns=adata.var_names)

pseudobulk_by_leiden = make_mean_pseudobulk(adata_mature, "leiden")
pseudobulk_by_sample = make_mean_pseudobulk(adata_mature, "sample")

pseudobulk_by_leiden.to_csv(TABLE_DIR / "mature_chp_pseudobulk_by_leiden.csv")
pseudobulk_by_sample.to_csv(TABLE_DIR / "mature_chp_pseudobulk_by_sample.csv")

print("Pseudobulk by Leiden:", pseudobulk_by_leiden.shape)
print("Pseudobulk by sample:", pseudobulk_by_sample.shape)
display(pseudobulk_by_leiden.head())

### Pseudobulk By Sample And Subcluster

A more specific summary groups cells by both developmental sample and mature ChP subcluster. This helps ask whether the same subcluster has similar expression at D27, D46, and D53, or whether a subcluster changes across stages.

In [ ]:
adata_mature.obs["sample_leiden"] = (
    adata_mature.obs["sample"].astype(str)
    + "__leiden_"
    + adata_mature.obs["leiden"].astype(str)
)

pseudobulk_by_sample_leiden = make_mean_pseudobulk(adata_mature, "sample_leiden")
pseudobulk_by_sample_leiden.to_csv(TABLE_DIR / "mature_chp_pseudobulk_by_sample_leiden.csv")

print("Pseudobulk by sample x Leiden:", pseudobulk_by_sample_leiden.shape)
display(pseudobulk_by_sample_leiden.head())

### Visualizing Pseudobulk Similarity

This correlation heatmap asks whether mature ChP subclusters are transcriptionally similar or distinct at the aggregate level. Closely related subclusters should show high correlation; divergent subclusters may represent specialized states.

In [ ]:
corr_by_leiden = pseudobulk_by_leiden.T.corr(method="pearson")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr_by_leiden, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(corr_by_leiden.shape[1]))
ax.set_xticklabels(corr_by_leiden.columns, rotation=90)
ax.set_yticks(range(corr_by_leiden.shape[0]))
ax.set_yticklabels(corr_by_leiden.index)
ax.set_title("Pseudobulk Correlation Between Mature ChP Subclusters")
plt.colorbar(im, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "mature_chp_pseudobulk_leiden_correlation.png", bbox_inches="tight", dpi=300)
plt.show()

## 3. Marker Extraction

Biological reasoning: markers help assign meaning to computational subclusters. For mature ChP cells, marker extraction can reveal whether a subcluster is ciliated/light, dark/mitochondria-rich, myoepithelial-like, barrier/transport-associated, early/developmental, or potentially stressed.

This section first tries to load marker genes saved from notebook 03. If they are unavailable, it reruns marker ranking on `adata_mature`.

In [ ]:
MATURE_MARKERS_PATH = PROCESSED_DIR / "mature_chp_subcluster_marker_genes.csv"

if MATURE_MARKERS_PATH.exists():
    mature_marker_table = pd.read_csv(MATURE_MARKERS_PATH)
    print("Loaded marker table:", MATURE_MARKERS_PATH)
else:
    print("Marker table not found; rerunning rank_genes_groups.")
    sc.tl.rank_genes_groups(adata_mature, groupby="leiden", method="wilcoxon")
    mature_marker_table = sc.get.rank_genes_groups_df(adata_mature, group=None)
    mature_marker_table.to_csv(MATURE_MARKERS_PATH, index=False)

display(mature_marker_table.head(20))

In [ ]:
top_n = 25
marker_genes_by_cluster = {
    str(cluster): (
        mature_marker_table[mature_marker_table["group"].astype(str) == str(cluster)]
        .sort_values("scores", ascending=False)
        .head(top_n)["names"]
        .dropna()
        .astype(str)
        .tolist()
    )
    for cluster in sorted(adata_mature.obs["leiden"].astype(str).unique(), key=lambda x: int(x) if x.isdigit() else x)
}

for cluster, genes in marker_genes_by_cluster.items():
    print(f"Cluster {cluster}: {genes[:10]}")

### Candidate Marker Sets Of Interest

These marker sets focus on biological programs observed in notebook 03:

- dark/mitochondria-rich ChP-like state
- light/ciliated ChP-like state
- barrier/transport functions
- myoepithelial-like identity
- stress response
- retinoic-acid-associated candidates
- RNA processing/splicing candidates such as `SRRM2`

These sets are not definitive cell-type labels. They are tools for interpreting patterns.

In [ ]:
candidate_marker_sets = {
    "Dark / mitochondria-rich ChP": ["CARD19", "IGF2", "RBP1"],
    "Light / ciliated ChP": ["FOXJ1", "ARL13B", "CCDC67"],
    "Barrier / transport": ["CLDN1", "CLDN3", "TJP1", "AQP1", "CA2", "SLC23A2"],
    "Myoepithelial-like": ["KRT17", "ACTA2", "TAGLN"],
    "Stress response": ["FOS", "JUN", "JUNB", "ATF3", "DDIT3", "HSPA1A", "HSPA1B"],
    "Retinoic-acid-associated candidates": ["CRABP2", "CRABP1", "RBP1", "ALDH1A1", "STRA6"],
    "RNA processing / splicing candidates": ["SRRM2", "SON", "SRSF1", "SRSF2", "SRSF3", "HNRNPA1", "HNRNPK", "SFPQ"],
}

candidate_marker_sets_present = {
    name: [gene for gene in genes if gene in adata_mature.var_names]
    for name, genes in candidate_marker_sets.items()
}
candidate_marker_sets_present = {
    name: genes for name, genes in candidate_marker_sets_present.items() if genes
}

for name, genes in candidate_marker_sets_present.items():
    print(f"{name}: {genes}")

sc.pl.dotplot(
    adata_mature,
    var_names=candidate_marker_sets_present,
    groupby="leiden",
    standard_scale="var",
    dendrogram=False,
)

## 4. Functional Enrichment

Biological reasoning: marker lists are hard to interpret gene-by-gene. Functional enrichment asks whether a marker list contains more genes from a known biological process than expected by chance.

Examples of relevant enriched functions might include:

- oxidative phosphorylation or mitochondrial translation for mitochondria-rich states
- cilium organization for light/ciliated states
- epithelial barrier and transport for ChP function
- extracellular matrix or contractile programs for myoepithelial-like states
- RNA processing/splicing for `SRRM2`-associated observations

This notebook includes two routes:

1. **Local fallback** using custom gene sets defined in this notebook.
2. **Optional gseapy/enrichr route** if `gseapy` is installed and internet access is available.

The local fallback is less comprehensive but is reproducible without internet.

In [ ]:
# Local, lightweight enrichment-style scoring by overlap with curated gene sets.
# This is not a replacement for GO/Reactome enrichment, but it helps prioritize themes.

local_gene_sets = {
    "mitochondrial_function": [
        "MT-CO1", "MT-CO2", "MT-CO3", "MT-ND1", "MT-ND2", "MT-ND3", "MT-ND4", "MT-ND5",
        "MT-CYB", "MT-ATP6", "MT-ATP8", "NDUFA1", "NDUFB8", "COX4I1", "ATP5F1A",
    ],
    "cilium": ["FOXJ1", "ARL13B", "CCDC67", "DNAH5", "PIFO", "TPPP3", "RSPH1"],
    "barrier_transport": ["CLDN1", "CLDN3", "CLDN5", "TJP1", "TJP2", "OCLN", "AQP1", "CA2", "SLC23A2"],
    "myoepithelial_contractile": ["KRT17", "ACTA2", "TAGLN", "MYL9", "CNN1"],
    "stress_response": ["FOS", "JUN", "JUNB", "ATF3", "DDIT3", "HSPA1A", "HSPA1B", "DNAJB1"],
    "retinoic_acid_related": ["CRABP2", "CRABP1", "RBP1", "RBP4", "ALDH1A1", "STRA6", "RARRES1", "RARRES2"],
    "rna_processing_splicing": ["SRRM2", "SON", "SRSF1", "SRSF2", "SRSF3", "SRSF7", "HNRNPA1", "HNRNPK", "SFPQ", "NONO"],
}

local_gene_sets_present = {
    name: set(gene for gene in genes if gene in adata_mature.var_names)
    for name, genes in local_gene_sets.items()
}

rows = []
for cluster, marker_genes in marker_genes_by_cluster.items():
    marker_set = set(marker_genes)
    for pathway, genes in local_gene_sets_present.items():
        overlap = sorted(marker_set & genes)
        rows.append({
            "cluster": cluster,
            "gene_set": pathway,
            "n_marker_genes": len(marker_set),
            "n_genes_in_set_present": len(genes),
            "n_overlap": len(overlap),
            "overlap_genes": ",".join(overlap),
        })

local_enrichment_summary = pd.DataFrame(rows)
local_enrichment_summary = local_enrichment_summary.sort_values(["cluster", "n_overlap"], ascending=[True, False])
local_enrichment_summary.to_csv(TABLE_DIR / "mature_chp_local_marker_gene_set_overlap.csv", index=False)
display(local_enrichment_summary.head(30))

In [ ]:
# Visualize local overlap counts.
overlap_matrix = local_enrichment_summary.pivot(
    index="cluster",
    columns="gene_set",
    values="n_overlap",
).fillna(0)

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(overlap_matrix, aspect="auto", cmap="magma")
ax.set_xticks(range(overlap_matrix.shape[1]))
ax.set_xticklabels(overlap_matrix.columns, rotation=45, ha="right")
ax.set_yticks(range(overlap_matrix.shape[0]))
ax.set_yticklabels(overlap_matrix.index)
ax.set_xlabel("Local gene set")
ax.set_ylabel("Mature ChP Leiden subcluster")
ax.set_title("Marker Gene Overlap With Curated Functional Themes")
plt.colorbar(im, ax=ax, label="Number of overlapping marker genes")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "mature_chp_local_enrichment_overlap_heatmap.png", bbox_inches="tight", dpi=300)
plt.show()

### Optional: Enrichr/GO Enrichment With `gseapy`

If `gseapy` is installed and internet access is available, this cell can run Enrichr using GO Biological Process terms. If it fails, continue with the local overlap analysis above and run formal enrichment later in a controlled environment.

In [ ]:
RUN_GSEAPY = False  # Change to True if gseapy is installed and internet access is available.

if RUN_GSEAPY:
    try:
        import gseapy as gp

        enrichr_results = []
        for cluster, genes in marker_genes_by_cluster.items():
            if len(genes) < 5:
                continue
            enr = gp.enrichr(
                gene_list=genes,
                gene_sets=["GO_Biological_Process_2023"],
                organism="Human",
                outdir=None,
            )
            result = enr.results.copy()
            result.insert(0, "cluster", cluster)
            enrichr_results.append(result)

        if enrichr_results:
            enrichr_table = pd.concat(enrichr_results, ignore_index=True)
            enrichr_table.to_csv(TABLE_DIR / "mature_chp_enrichr_go_bp_results.csv", index=False)
            display(enrichr_table.head(30))
        else:
            print("No Enrichr results generated.")
    except Exception as exc:
        print("gseapy/Enrichr analysis failed:", exc)
else:
    print("Skipping gseapy. Set RUN_GSEAPY = True to run optional Enrichr analysis.")

## 5. Dark Vs Light Subtype Comparison

Biological reasoning: the paper discusses specialized mature ChP epithelial states, including dark/mitochondria-rich and light/ciliated ChP-like populations. Comparing these programs helps ask whether they represent distinct axes of mature ChP specialization.

Here, we use signature scores from notebook 03 if available. If they are missing, we calculate them again.

In [ ]:
score_definitions = {
    "dark_mito_chp_score": ["CARD19", "IGF2", "RBP1"],
    "light_ciliated_chp_score": ["FOXJ1", "ARL13B", "CCDC67"],
    "barrier_transport_score": ["CLDN1", "CLDN3", "TJP1", "AQP1", "CA2", "SLC23A2"],
    "mitochondrial_gene_score": ["MT-CO1", "MT-CO2", "MT-CO3", "MT-ND1", "MT-ND2", "MT-ND3", "MT-ND4", "MT-ND5", "MT-CYB"],
}

for score_name, genes in score_definitions.items():
    if score_name in adata_mature.obs.columns:
        continue
    genes_present = [gene for gene in genes if gene in adata_mature.var_names]
    if genes_present:
        sc.tl.score_genes(adata_mature, gene_list=genes_present, score_name=score_name)
        print(f"Added {score_name}: {genes_present}")
    else:
        print(f"No genes present for {score_name}")

available_scores = [score for score in score_definitions if score in adata_mature.obs.columns]
print("Available scores:", available_scores)

In [ ]:
if {"dark_mito_chp_score", "light_ciliated_chp_score"}.issubset(adata_mature.obs.columns):
    fig, ax = plt.subplots(figsize=(6, 5))
    scatter = ax.scatter(
        adata_mature.obs["dark_mito_chp_score"],
        adata_mature.obs["light_ciliated_chp_score"],
        c=adata_mature.obs["leiden"].astype("category").cat.codes,
        cmap="tab20",
        s=6,
        alpha=0.7,
        linewidths=0,
    )
    ax.set_xlabel("Dark / mitochondria-rich ChP score")
    ax.set_ylabel("Light / ciliated ChP score")
    ax.set_title("Dark vs Light Mature ChP Programs")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "mature_chp_dark_vs_light_score_scatter.png", bbox_inches="tight", dpi=300)
    plt.show()
else:
    print("Dark/light score columns are not available.")

In [ ]:
score_summary_by_leiden = adata_mature.obs.groupby("leiden", observed=True)[available_scores].mean()
display(score_summary_by_leiden)
score_summary_by_leiden.to_csv(TABLE_DIR / "mature_chp_dark_light_scores_by_leiden.csv")

if available_scores:
    fig, ax = plt.subplots(figsize=(10, 5))
    im = ax.imshow(score_summary_by_leiden, aspect="auto", cmap="viridis")
    ax.set_xticks(range(score_summary_by_leiden.shape[1]))
    ax.set_xticklabels(score_summary_by_leiden.columns, rotation=45, ha="right")
    ax.set_yticks(range(score_summary_by_leiden.shape[0]))
    ax.set_yticklabels(score_summary_by_leiden.index)
    ax.set_xlabel("Signature score")
    ax.set_ylabel("Mature ChP Leiden subcluster")
    ax.set_title("Dark/Light/Barrier/Mito Scores By Subcluster")
    plt.colorbar(im, ax=ax, label="Mean score")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "mature_chp_dark_light_scores_heatmap.png", bbox_inches="tight", dpi=300)
    plt.show()

### Dark Vs Light Interpretation Notes

Use this section to record whether dark and light programs appear mutually exclusive, overlapping, or weakly separated. If one cluster has high mitochondrial gene score but low dark ChP score, that suggests broad mitochondrial expression is not the same as the specific dark ChP marker program.

## 6. Sample-Stage Composition Analysis

Biological reasoning: the ChP organoid samples represent developmental stages D27, D46, and D53. If a mature ChP subcluster is enriched at D27 and reduced by D53, it may represent an earlier or transitional state. If a subcluster increases by D53, it may represent a more mature or specialized state.

This analysis asks whether mature ChP subclusters are evenly distributed across stages or stage-enriched.

In [ ]:
sample_counts_by_leiden = pd.crosstab(
    adata_mature.obs["leiden"],
    adata_mature.obs["sample"],
)

sample_percent_by_leiden = pd.crosstab(
    adata_mature.obs["leiden"],
    adata_mature.obs["sample"],
    normalize="index",
) * 100
sample_percent_by_leiden = sample_percent_by_leiden.round(2)

sample_counts_by_leiden.to_csv(TABLE_DIR / "mature_chp_sample_counts_by_leiden.csv")
sample_percent_by_leiden.to_csv(TABLE_DIR / "mature_chp_sample_percent_by_leiden.csv")

display(sample_counts_by_leiden)
display(sample_percent_by_leiden)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(sample_percent_by_leiden, aspect="auto", cmap="viridis")
ax.set_xticks(range(sample_percent_by_leiden.shape[1]))
ax.set_xticklabels(sample_percent_by_leiden.columns.astype(str), rotation=45, ha="right")
ax.set_yticks(range(sample_percent_by_leiden.shape[0]))
ax.set_yticklabels(sample_percent_by_leiden.index.astype(str))
ax.set_xlabel("Sample")
ax.set_ylabel("Mature ChP Leiden subcluster")
ax.set_title("Sample Composition Of Mature ChP Subclusters")
plt.colorbar(im, ax=ax, label="Percent of subcluster cells")

for i in range(sample_percent_by_leiden.shape[0]):
    for j in range(sample_percent_by_leiden.shape[1]):
        value = sample_percent_by_leiden.iloc[i, j]
        text_color = "white" if value > 50 else "black"
        ax.text(j, i, f"{value:.1f}", ha="center", va="center", color=text_color, fontsize=8)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "mature_chp_sample_stage_composition_heatmap.png", bbox_inches="tight", dpi=300)
plt.show()

### Stage-Specific Interpretation Notes

Use the heatmap to identify candidate stage-enriched states. For example, a D27-enriched cluster may represent an earlier mature-ChP-like state, while D53-enriched clusters may represent more specialized mature ChP states. Stage enrichment should be interpreted together with marker genes and signature scores, not by sample composition alone.

## 7. Interpretation

This section should be completed after running the notebook. Suggested points to address:

- Which mature ChP subclusters are dark/mitochondria-rich candidates?
- Which are light/ciliated candidates?
- Which clusters show barrier/transport programs?
- Does broad mitochondrial gene expression match the dark ChP marker program, or do these separate?
- Which subclusters are enriched at D27, D46, or D53?
- Does cluster 10 represent an early/less-specialized mature ChP-like state?
- Is `CRABP2` a cluster-specific candidate marker or part of a broader retinoic-acid-associated program?
- Does `SRRM2` appear alone or with a broader RNA-processing/splicing signature?

Careful interpretation is important because the input matrix is SCT-scaled and the analysis is exploratory. Functional enrichment and reference comparison should be used to generate hypotheses, not final disease claims.

### Draft Interpretation Template

After running the notebook, replace this template with your results:

```text
Mature ChP subclusters showed [shared/distinct] transcriptional programs. Subcluster(s) [X] showed dark/mitochondria-rich ChP marker enrichment, while subcluster(s) [Y] showed light/ciliated marker enrichment. Barrier/transport scores were highest in [clusters], suggesting [interpretation]. Sample-stage composition indicated that [cluster] was enriched at D27 and [cluster] was enriched at D53, consistent with possible developmental progression. Functional enrichment/overlap analysis suggested [pathways]. These observations support the hypothesis that mature ChP organoids contain specialized epithelial states, while also identifying candidate early or transitional states for future reference comparison.
```